In [0]:

from pyspark.sql.functions import col

# Step 1: Read Bronze customers
df_customers = spark.table(
    "retail_analytics_catalog.retail_analytics.bronze_customers"
)

# Step 2: Remove records missing important customer IDs
silver_customers = df_customers.filter(
    col("customer_id").isNotNull() &
    col("customer_unique_id").isNotNull()
)

# Step 3: Remove duplicate customer IDs
silver_customers = silver_customers.dropDuplicates(["customer_id"])

# Step 4: Save cleaned data as a Silver Delta table
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_analytics_catalog.retail_analytics.silver_customers"
    )

print("Silver customers table created successfully")


Silver customers table created successfully


In [0]:

from pyspark.sql.functions import col, lower, trim

# Step 1: Read Bronze Orders
df_orders = spark.table(
    "retail_analytics_catalog.retail_analytics.bronze_orders"
)

# Step 2: Remove records missing important IDs
silver_orders = df_orders.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull()
)

# Step 3: Standardize order status and remove duplicates
silver_orders = (
    silver_orders
    .withColumn("order_status", lower(trim(col("order_status"))))
    .dropDuplicates(["order_id"])
)

# Step 4: Save as Silver Delta table
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_analytics_catalog.retail_analytics.silver_orders"
    )

print("Silver orders table created successfully")


Silver orders table created successfully


In [0]:

from pyspark.sql.functions import col

# Step 1: Read Bronze Order Items
df_order_items = spark.table(
    "retail_analytics_catalog.retail_analytics.bronze_order_items"
)

# Step 2: Remove records missing important IDs
silver_order_items = df_order_items.filter(
    col("order_id").isNotNull() &
    col("order_item_id").isNotNull() &
    col("product_id").isNotNull()
)

# Step 3: Keep valid prices and shipping costs
silver_order_items = silver_order_items.filter(
    (col("price") >= 0) &
    (col("freight_value") >= 0)
)

# Step 4: Remove duplicate order items
silver_order_items = silver_order_items.dropDuplicates(
    ["order_id", "order_item_id"]
)

# Step 5: Save as Silver Delta table
silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_analytics_catalog.retail_analytics.silver_order_items"
    )

print("Silver order items table created successfully")


Silver order items table created successfully


In [0]:

from pyspark.sql.functions import col, lower, trim

# Step 1: Read Bronze Products
df_products = spark.table(
    "retail_analytics_catalog.retail_analytics.bronze_products"
)

# Step 2: Remove records without product_id
silver_products = df_products.filter(
    col("product_id").isNotNull()
)

# Step 3: Standardize product category names
silver_products = silver_products.withColumn(
    "product_category_name",
    lower(trim(col("product_category_name")))
)

# Step 4: Remove duplicate products
silver_products = silver_products.dropDuplicates(["product_id"])

# Step 5: Save as Silver Delta table
silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_analytics_catalog.retail_analytics.silver_products"
    )

print("Silver products table created successfully")


Silver products table created successfully


In [0]:

tables = [
    "silver_customers",
    "silver_orders",
    "silver_order_items",
    "silver_products"
]

for table in tables:
    full_table_name = f"retail_analytics_catalog.retail_analytics.{table}"
    count = spark.table(full_table_name).count()
    print(f"{table}: {count} records")


silver_customers: 99441 records
silver_orders: 99441 records
silver_order_items: 112650 records
silver_products: 32951 records
